libraries

In [1]:
import yaml
import pandas as pd
from datetime import date
from sqlalchemy import create_engine


Connection

In [3]:
# Cargar la configuración externa del proyecto
with open("../config_fill.yml", "r") as f:
    config = yaml.safe_load(f)

# Configurar motores de conexión usando SQLAlchemy
c_orig = config['CO_SA']
url_origen = f"{c_orig['drivername']}://{c_orig['user']}:{c_orig['password']}@{c_orig['host']}:{c_orig['port']}/{c_orig['dbname']}"
engine_origen = create_engine(url_origen)

c_dest = config['ETL_PRO']
url_destino = f"{c_dest['drivername']}://{c_dest['user']}:{c_dest['password']}@{c_dest['host']}:{c_dest['port']}/{c_dest['dbname']}"
engine_destino = create_engine(url_destino)


Extract

In [4]:
try:
    print("====== FASE: EXTRACT ======")
    df_cliente = pd.read_sql_table('cliente', engine_origen)
    print(f"📊 Registros extraídos del origen -> Clientes: {len(df_cliente)}")
except Exception as e:
    print("❌ Error en la extracción:")
    print(e)

====== FASE: EXTRACT ======
📊 Registros extraídos del origen -> Clientes: 27


Transform

In [5]:
try:
    print("====== FASE: TRANSFORM ======")
    
    # 1. Filtrar únicamente las columnas definidas en el diagrama del modelo estrella
    df_dim_cliente = df_cliente[[
        'cliente_id', 'nombre', 'telefono', 'email', 'nit_cliente'
    ]].copy()
    
    # 2. Renombrar las columnas EXACTAMENTE igual a las del diagrama conceptual
    df_dim_cliente.rename(columns={
        'cliente_id': 'ID_Cliente',
        'nombre': 'Nombre_Empresa',
        'telefono': 'Telefono',
        'email': 'Correo_Electronico',
        'nit_cliente': 'NIT_Cedula'
    }, inplace=True)
    
    # 3. Tratamiento de nulos para garantizar la calidad en los reportes
    df_dim_cliente['Nombre_Empresa'] = df_dim_cliente['Nombre_Empresa'].fillna('No especificado')
    df_dim_cliente['Telefono'] = df_dim_cliente['Telefono'].fillna('Sin Registro')
    df_dim_cliente['Correo_Electronico'] = df_dim_cliente['Correo_Electronico'].fillna('Sin Correo')
    df_dim_cliente['NIT_Cedula'] = df_dim_cliente['NIT_Cedula'].fillna('Sin Identificación')
    
    # 4. Auditoría técnica (Estándar de linaje de datos del grupo)
    df_dim_cliente["saved"] = date.today()
    
    # 5. Generación de Clave Subrogada (Garantiza indexación óptima en el Data Warehouse)
    df_dim_cliente.insert(0, 'id_dim_cliente', range(1, len(df_dim_cliente) + 1))
    
    print("✅ Transformación estructurada completada con éxito.")
    display(df_dim_cliente.head())
except Exception as e:
    print("❌ Error en la transformación:")
    print(e)

====== FASE: TRANSFORM ======
✅ Transformación estructurada completada con éxito.


,id_dim_cliente,ID_Cliente,Nombre_Empresa,Telefono,Correo_Electronico,NIT_Cedula,saved
0,1,1,Cliente 2,327-00000,algo.com,25,2026-06-24
1,2,2,Cliente 1,327-00000,algo.com,123,2026-06-24
2,3,6,CLINICA DEPORTIVA DEL SUR,327-00000,algo.com,24390-3,2026-06-24
3,4,19,HOSPITAL ORTOPEDICO DE COLOMBIA,327-00000,algo.com,8301821,2026-06-24
4,5,8,CLINICA NEFROLOGOS DE CALI,327-00000,algo.com,5017350-8,2026-06-24


Load

In [6]:
try:
    print("====== FASE: LOAD ======")
    nombre_tabla_olap = 'dim_cliente'
    
    # Carga física en la base de datos destino 'etl_project'
    df_dim_cliente.to_sql(
        name=nombre_tabla_olap, 
        con=engine_destino, 
        if_exists='replace', 
        index_label='key_dim_cliente'
    )
    
    print(f"✅ ¡PROCESO CONCLUIDO! La dimensión '{nombre_tabla_olap}' está lista y conectada al modelo.")
except Exception as e:
    print("❌ Error en la carga de datos:")
    print(e)

====== FASE: LOAD ======
✅ ¡PROCESO CONCLUIDO! La dimensión 'dim_cliente' está lista y conectada al modelo.
